In [8]:
import random
import numpy as np
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
from scripts.cleaning_gpt_data import get_raw_data, process_txt_dataset, extract_personal_info, all_questionnaires, process_associations, find_missing_cue_words

## GPT-3.5

In [3]:
abs_path = '/Users/emma/projects/cognoscolab/math_anxiety_LLMs'

In [4]:
folder_gpt_3_5 = f'{abs_path}/data/gpt_3_5_students_data'

raw_data_gpt_3_5 = get_raw_data(folder_gpt_3_5)
cleaned_data_gpt_3_5 = process_txt_dataset(raw_data_gpt_3_5, 'gpt_3_5')
personal_data_gpt_3_5 = extract_personal_info(cleaned_data_gpt_3_5)
all_questionnaires_gpt_3_5, mas_it_gpt_3_5 = all_questionnaires(personal_data_gpt_3_5)
association_valence_gpt_3_5 = process_associations(cleaned_data_gpt_3_5)
find_missing_cue_words(association_valence_gpt_3_5, personal_data_gpt_3_5)

Number of generated files: 300
Number of cue words: 40
Number of participants: 300
No missing cue words ✅


## GPT-4o

In [5]:
folder_gpt_4o = f'{abs_path}/data/gpt_4o_students_data'

raw_data_gpt_4o = get_raw_data(folder_gpt_4o)
cleaned_data_gpt_4o = process_txt_dataset(raw_data_gpt_4o, 'gpt_4o')
personal_data_gpt_4o = extract_personal_info(cleaned_data_gpt_4o)
all_questionnaires_gpt_4o, mas_it_gpt_4o = all_questionnaires(personal_data_gpt_4o)
association_valence_gpt_4o = process_associations(cleaned_data_gpt_4o)
find_missing_cue_words(association_valence_gpt_4o, personal_data_gpt_4o)

Number of generated files: 356
Number of cue words: 40
Number of participants: 356
No missing cue words ✅


## Remove participats to have 300 students for GPT-4o

In [32]:
# all the rows with empty values in the association task
empty_row_associations = association_valence_gpt_4o[association_valence_gpt_4o.isnull().any(axis=1)]
print(f"Participant with some empty rows in the associations task: {empty_row_associations['participant_id'].unique()}")
# all the rows with empty values in the mas-it task
empty_rows_mas_it = mas_it_gpt_4o[mas_it_gpt_4o.isnull().any(axis=1)]
print(f"Participant with some empty rows in the mas-it: {empty_rows_mas_it['participant_id'].unique()}")

# combine participants with empty rows in either associations OR mas_it
to_remove = np.union1d(
    empty_row_associations['participant_id'].unique(),
    empty_rows_mas_it['participant_id'].unique()
)
# remove from the associations df the participants with empty rows in the associations task and in the mas-it task
association_valence_gpt_4o_cleaned = association_valence_gpt_4o[~association_valence_gpt_4o['participant_id'].isin(to_remove)]

mas_it_gpt_4o_cleaned = mas_it_gpt_4o[~mas_it_gpt_4o['participant_id'].isin(to_remove)]

# creates a list of participant_ids left after removing the ones with empty rows
participant_ids_left = mas_it_gpt_4o_cleaned['participant_id']

random.seed(42)  # for reproducibility
# randomly select 44 participant_ids to remove from the ones left -> to get 300 students
participant_ids_to_remove = random.sample(list(participant_ids_left), len(participant_ids_left) - 300)
print(f"Randomly selected participant_ids to remove: {participant_ids_to_remove}")

# remove the selected participant_ids from the associations and mas-it df
association_valence_gpt_4o_cleaned = association_valence_gpt_4o_cleaned[~association_valence_gpt_4o_cleaned['participant_id'].isin(participant_ids_to_remove)]

mas_it_gpt_4o_cleaned = mas_it_gpt_4o_cleaned[~mas_it_gpt_4o_cleaned['participant_id'].isin(participant_ids_to_remove)]

all_questionnaires_gpt_4o_cleaned = all_questionnaires_gpt_4o[~all_questionnaires_gpt_4o['participant_id'].isin(np.concatenate([participant_ids_to_remove, to_remove]))]

Participant with some empty rows in the associations task: ['gpt_4o_301' 'gpt_4o_073' 'gpt_4o_154' 'gpt_4o_115' 'gpt_4o_271'
 'gpt_4o_066']
Participant with some empty rows in the mas-it: ['gpt_4o_059' 'gpt_4o_082' 'gpt_4o_089' 'gpt_4o_134' 'gpt_4o_168'
 'gpt_4o_184']
Randomly selected participant_ids to remove: ['gpt_4o_340', 'gpt_4o_058', 'gpt_4o_013', 'gpt_4o_148', 'gpt_4o_132', 'gpt_4o_121', 'gpt_4o_075', 'gpt_4o_053', 'gpt_4o_291', 'gpt_4o_045', 'gpt_4o_315', 'gpt_4o_227', 'gpt_4o_017', 'gpt_4o_016', 'gpt_4o_048', 'gpt_4o_118', 'gpt_4o_126', 'gpt_4o_269', 'gpt_4o_321', 'gpt_4o_014', 'gpt_4o_299', 'gpt_4o_107', 'gpt_4o_345', 'gpt_4o_225', 'gpt_4o_119', 'gpt_4o_240', 'gpt_4o_314', 'gpt_4o_150', 'gpt_4o_004', 'gpt_4o_086', 'gpt_4o_185', 'gpt_4o_084', 'gpt_4o_117', 'gpt_4o_182', 'gpt_4o_205', 'gpt_4o_050', 'gpt_4o_194', 'gpt_4o_187', 'gpt_4o_322', 'gpt_4o_143', 'gpt_4o_023', 'gpt_4o_246', 'gpt_4o_286', 'gpt_4o_065']


# Exporting to csv

In [33]:
association_valence_gpt_3_5.to_csv(f'{abs_path}/out/formatting/gpt_3_5/associations_valence_gpt_3_5.csv', index=False)

all_questionnaires_gpt_3_5.to_csv(f'{abs_path}/out/formatting/gpt_3_5/all_questionnaires_gpt_3_5.csv', index=False)

mas_it_gpt_3_5.to_csv(f'{abs_path}/out/formatting/gpt_3_5/mas_it_gpt_3_5.csv', index=False)

In [34]:
association_valence_gpt_4o_cleaned.to_csv(f'{abs_path}/out/formatting/gpt_4o/associations_valence_gpt_4o.csv', index= False)

all_questionnaires_gpt_4o_cleaned.to_csv(f'{abs_path}/out/formatting/gpt_4o/all_questionnaires_gpt_4o.csv', index= False)

mas_it_gpt_4o_cleaned.to_csv(f'{abs_path}/out/formatting/gpt_4o/mas_it_gpt_4o.csv', index= False)